In [16]:
import cv2
from ultralytics import YOLO
import numpy as np

#intränade modellen
model = YOLO("runs/detect/roskispolis3/weights/best.pt")

#max distans mellan objekt inom samma ruta
max_distance = 175

#klass namn
class_names = ["dryckeskartong", "konservburk", "pantburk"]

#videofilen
video_path = "VIDEOFILEN2.mp4"

#öppnar videon
cap = cv2.VideoCapture(video_path)

#checkar om videon öppnas 
if not cap.isOpened():
    print("Error opening video file")
    exit()

#räknar mittpunkten av lådan
def get_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) // 2, (y1 + y2) // 2)

#räknar distansen mellan 2 lådor
def distance(p1,p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))
    
#loopar igenom videon frame by frame
while cap.isOpened():
        #läser en frame av videon
        ret, frame = cap.read()
        #stoppar videon om inga frames 
        if not ret:
            break

        #gör YOLO modellen på bilden
        results = model(frame)[0]

        detections = []

        #loopar igenom detekterade objekt
        for box, cls in zip(results.boxes.xyxy, results.boxes.cls):
            #konverterar bounding box till en integer
            x1, y1, x2, y2 = map(int, box)
            #konverterar klass index till en integer
            cls = int(cls)
            #räknar mittpunkten av bounding boxen
            center = get_center((x1, y1, x2, y2))

            #sparar detektionen till en lista
            detections.append({
            "box": (x1, y1, x2, y2),
            "class": cls,
            "center": center
            })

        #normalt läge
        states = ["alone"] * len(detections)

        #jämför alla objekt med alla andra objekt
        for i in range(len(detections)):
            for j in range(i + 1, len(detections)):

                #räknar distansen mellan två objekt
                d = distance(detections[i]["center"], detections[j]["center"])

                #om distansen är mindre än max distance och klassen är samma blir de "sorted" annars är de osorterade
                #om distansen är längre blir de gula
                if d < max_distance:
                    #om de är olika klasser blir de osorterade
                    if detections[i]["class"] != detections[j]["class"]:
                        states[i] = "unsorted"
                        states[j] = "unsorted"
                    else:
                        #om klassen är samma och de inte redan är osorterade blir de sorterade
                        if states[i] != "unsorted":
                            states[i] = "sorted"
                        if states[j] != "unsorted":
                            states[j] = "sorted"


        #rita resultat
        for i, det in enumerate(detections):
            x1, y1, x2, y2 = det["box"]
            cls = det["class"]
            #får klassnamnet
            label = class_names[cls]

            if states[i] == "sorted":
                color = (0, 255, 0) #grön
                text = label.upper()
            elif states[i] == "unsorted":
                color = (0, 0, 255) #röd
                text = label
            else:
                color = (0, 255, 255) #gul
                text = label

            #ritar bounding boxen
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, text, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.imshow("sorting detection", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()


0: 640x384 1 Konservburk, 1 Pantburk, 44.9ms
Speed: 9.9ms preprocess, 44.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Konservburk, 1 Pantburk, 47.8ms
Speed: 1.7ms preprocess, 47.8ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Konservburk, 1 Pantburk, 44.8ms
Speed: 1.5ms preprocess, 44.8ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Konservburk, 1 Pantburk, 43.0ms
Speed: 1.4ms preprocess, 43.0ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Konservburk, 1 Pantburk, 43.2ms
Speed: 1.5ms preprocess, 43.2ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Konservburk, 1 Pantburk, 44.6ms
Speed: 1.4ms preprocess, 44.6ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 Konservburk, 1 Pantburk, 39.1ms
Speed: 2.1ms preprocess, 39.1ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)